# Fruit and Vegetable Classification (Deep Learning, Google Colab)

A CNN (transfer learning with **MobileNetV2**) that classifies images of fruits and vegetables.

Steps: download dataset -> explore -> build data pipeline -> train -> fine-tune -> evaluate -> predict on your own image.

**Tip:** Use a GPU: *Runtime > Change runtime type > T4 GPU*. Then run cells from top to bottom.

## 1. Imports and GPU check

In [ ]:
import os, json, random, shutil, zipfile
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2. Download and extract the dataset (Google Drive)

In [ ]:
%pip install -q gdown
import gdown

FILE_ID = "1CGiAWso43GCsNo_faRq4jdDIlmwy7YI4"
ZIP_PATH = "Fruits_Vegetables.zip"
EXTRACT_DIR = "/content/data"

if not os.path.exists(ZIP_PATH):
    gdown.download(id=FILE_ID, output=ZIP_PATH, quiet=False)

if not os.path.exists(ZIP_PATH):
    raise SystemExit("Download failed. Upload Fruits_Vegetables.zip manually using the Files panel on the left, then re-run.")

os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(EXTRACT_DIR)
print("Extracted to", EXTRACT_DIR)

## 3. Explore the folder structure and remove corrupted images

In [ ]:
from PIL import Image

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# Remove unreadable images (they would crash training)
removed = 0
for root, _, files in os.walk(EXTRACT_DIR):
    if "__MACOSX" in root:
        continue
    for f in files:
        p = os.path.join(root, f)
        if f.lower().endswith(IMG_EXT):
            try:
                with Image.open(p) as im:
                    im.verify()
            except Exception:
                os.remove(p); removed += 1
        elif f.startswith("._") or f.lower() in (".ds_store", "thumbs.db"):
            os.remove(p)
print("Corrupted images removed:", removed)

def count_images(d):
    return sum(f.lower().endswith(IMG_EXT) for f in os.listdir(d)
               if os.path.isfile(os.path.join(d, f)))

# A "class root" = a folder whose sub-folders directly contain images (one sub-folder per class)
class_roots = {}
for root, dirs, _ in os.walk(EXTRACT_DIR):
    if "__MACOSX" in root:
        continue
    subs = [d for d in dirs if count_images(os.path.join(root, d)) > 0]
    if len(subs) >= 2:
        class_roots[root] = len(subs)

for r, n in class_roots.items():
    total = sum(count_images(os.path.join(r, d)) for d in os.listdir(r) if os.path.isdir(os.path.join(r, d)))
    print(f"{r}\n   classes: {n}, images: {total}")

## 4. Detect train / validation / test folders
The code below tries to find them automatically. If your zip has a different layout,
set `TRAIN_DIR`, `VAL_DIR`, `TEST_DIR` (or `SINGLE_DIR`) by hand at the end of this cell.

In [ ]:
TRAIN_DIR = VAL_DIR = TEST_DIR = SINGLE_DIR = None

for r in class_roots:
    name = os.path.basename(r).lower()
    if name == "train":
        TRAIN_DIR = r
    elif name in ("validation", "valid", "val"):
        VAL_DIR = r
    elif name == "test":
        TEST_DIR = r

if TRAIN_DIR is None:
    # No train/val/test split: use the biggest class folder and split it ourselves
    SINGLE_DIR = max(class_roots, key=class_roots.get)

# ---- Manual override (uncomment and edit if auto-detection is wrong) ----
# TRAIN_DIR = "/content/data/.../train"
# VAL_DIR   = "/content/data/.../validation"
# TEST_DIR  = "/content/data/.../test"
# SINGLE_DIR = None

print("TRAIN :", TRAIN_DIR)
print("VAL   :", VAL_DIR)
print("TEST  :", TEST_DIR)
print("SINGLE:", SINGLE_DIR)

## 5. Build the data pipeline

In [ ]:
IMG_SIZE = (224, 224)
BATCH = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

def load(d, **kw):
    return tf.keras.utils.image_dataset_from_directory(
        d, image_size=IMG_SIZE, batch_size=BATCH, seed=SEED, **kw)

if TRAIN_DIR:
    if VAL_DIR:
        train_ds = load(TRAIN_DIR)
        val_ds = load(VAL_DIR, shuffle=False)
    else:
        train_ds = load(TRAIN_DIR, validation_split=0.2, subset="training")
        val_ds = load(TRAIN_DIR, validation_split=0.2, subset="validation")
    test_ds = load(TEST_DIR, shuffle=False) if TEST_DIR else val_ds
else:
    train_ds = load(SINGLE_DIR, validation_split=0.3, subset="training")
    rest = load(SINGLE_DIR, validation_split=0.3, subset="validation")
    n = tf.data.experimental.cardinality(rest).numpy()
    val_ds = rest.take(n // 2)
    test_ds = rest.skip(n // 2)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Number of classes:", num_classes)
print(class_names)

# Show some training images
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.show()

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## 6. Build the model (MobileNetV2 transfer learning)

In [ ]:
data_aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="augmentation")

base = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False  # freeze first

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = data_aug(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)
model = models.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

## 7. Train (feature extraction)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)

## 8. Fine-tune the top layers of MobileNetV2 (improves accuracy)

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

history_ft = model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=callbacks)

## 9. Training curves

In [ ]:
acc = history.history["accuracy"] + history_ft.history["accuracy"]
val_acc = history.history["val_accuracy"] + history_ft.history["val_accuracy"]
loss = history.history["loss"] + history_ft.history["loss"]
val_loss = history.history["val_loss"] + history_ft.history["val_loss"]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(acc, label="train"); ax[0].plot(val_acc, label="val"); ax[0].set_title("Accuracy"); ax[0].legend()
ax[1].plot(loss, label="train"); ax[1].plot(val_loss, label="val"); ax[1].set_title("Loss"); ax[1].legend()
plt.show()

## 10. Evaluate on the test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

y_true, y_pred = [], []
for xb, yb in test_ds:
    y_pred.extend(np.argmax(model.predict(xb, verbose=0), axis=1))
    y_true.extend(yb.numpy())

print("Test accuracy: %.4f" % np.mean(np.array(y_true) == np.array(y_pred)))
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

size = max(6, num_classes * 0.35)
fig, ax = plt.subplots(figsize=(size, size))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=class_names).plot(
    ax=ax, xticks_rotation=90, colorbar=False, include_values=num_classes <= 15)
plt.show()

## 11. Sample predictions

In [ ]:
plt.figure(figsize=(12, 8))
for images, labels in test_ds.take(1):
    preds = np.argmax(model.predict(images, verbose=0), axis=1)
    for i in range(min(12, len(images))):
        plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        ok = preds[i] == labels[i].numpy()
        plt.title(f"{class_names[preds[i]]}\n(true: {class_names[labels[i]]})",
                  color="green" if ok else "red", fontsize=9)
        plt.axis("off")
plt.tight_layout(); plt.show()

## 12. Save the model

In [ ]:
model.save("fruit_veg_model.keras")
with open("class_names.json", "w") as f:
    json.dump(class_names, f)
print("Saved fruit_veg_model.keras and class_names.json")

try:
    from google.colab import files
    files.download("fruit_veg_model.keras")
    files.download("class_names.json")
except Exception:
    pass

## 13. Try it on your own image
Run the cell and choose a fruit/vegetable photo from your computer.

In [ ]:
from google.colab import files

uploaded = files.upload()
for name in uploaded:
    img = tf.keras.utils.load_img(name, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)[None, ...]
    probs = model.predict(arr, verbose=0)[0]
    top = np.argsort(probs)[::-1][:3]
    plt.imshow(img); plt.axis("off")
    plt.title(f"{class_names[top[0]]} ({probs[top[0]]*100:.1f}%)")
    plt.show()
    for i in top:
        print(f"{class_names[i]:<25} {probs[i]*100:.2f}%")